In [2]:
# Import required libraries used for dataset inspection and creation.

import pandas as pd
import ast

In [3]:
# Load the original Amazon Electronics dataset.
# The raw file will never be modified.

raw_path = "../raw/Amazon Electronics Metadata.csv"

df = pd.read_csv(raw_path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (498196, 9)


In [4]:
# Display the basic structure of the raw dataset.

print("Columns:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

Columns:
['asin', 'imUrl', 'description', 'categories', 'title', 'price', 'salesRank', 'related', 'brand']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 498196 entries, 0 to 498195
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   asin         498196 non-null  object 
 1   imUrl        498021 non-null  object 
 2   description  442139 non-null  object 
 3   categories   498196 non-null  object 
 4   title        491192 non-null  object 
 5   price        389693 non-null  float64
 6   salesRank    128706 non-null  object 
 7   related      366959 non-null  object 
 8   brand        141365 non-null  object 
dtypes: float64(1), object(8)
memory usage: 34.2+ MB


In [5]:
# Check missing values in each column.

missing_values = df.isnull().sum()

print("Missing values:")
print(missing_values)

print("\nMissing percentage:")
missing_percentage = (df.isnull().sum() / len(df)) * 100
print(missing_percentage.round(2))

Missing values:
asin                0
imUrl             175
description     56057
categories          0
title            7004
price          108503
salesRank      369490
related        131237
brand          356831
dtype: int64

Missing percentage:
asin            0.00
imUrl           0.04
description    11.25
categories      0.00
title           1.41
price          21.78
salesRank      74.17
related        26.34
brand          71.62
dtype: float64


In [6]:
# Inspect the price 

print(df["price"].describe())

print("\nNegative prices:", (df["price"] < 0).sum())
print("Zero prices:", (df["price"] == 0).sum())

count    389693.000000
mean         61.406786
std         119.118870
min           0.010000
25%           9.950000
50%          19.990000
75%          51.950000
max         999.990000
Name: price, dtype: float64

Negative prices: 0
Zero prices: 0


In [3]:
# keep only rows that app actually needs
required_fields = ["asin", "title", "description", "categories", "brand", "price", "imUrl"]

valid_products = df.dropna(subset=required_fields).copy()

valid_products = valid_products[
    (valid_products["title"].astype(str).str.strip() != "") &
    (valid_products["description"].astype(str).str.strip() != "")].copy()

# price floor value 1$ 
valid_products = valid_products[valid_products["price"] > 1].copy()

print("Valid products after field/price filtering:", len(valid_products))


Valid products after field/price filtering: 124518


In [5]:
# category classification

PRODUCT_TYPE_KEYWORDS = {
    "headphone": ["headphone", "earphone", "earbud", "headset"],
    "laptop": ["laptop", "notebook computer"],
    "phone": ["smartphone", "cell phone"],
    "tablet": ["tablet", "ipad"],
    "keyboard": ["keyboard"],
    "mouse": ["mouse"],
}

EXCLUDED_KEYWORDS = [
    "case", "cover", "battery", "cable", "adapter", "adaptor",
    "mount", "stand", "screen protector", "replacement screen",
    "replacement part", "replacement", "bag", "holder", "dock",
    "sleeve", "protective", "mouse pad", "remote control", "flash",
    "lens", "tripod", "bracket", "converter", "splitter",
    "receiver", "transmitter", "privacy screen", "privacy filter",
    "antiglare screen", "charger", "charging", "charging cable",
]

def get_product_type(title):
    
    title = str(title).lower()

    for product_type, keywords in PRODUCT_TYPE_KEYWORDS.items():
        if any(keyword in title for keyword in keywords):
            return product_type

    return None


def is_excluded(title):
    title = str(title).lower()
    return any(keyword in title for keyword in EXCLUDED_KEYWORDS)


# classification

valid_products["product_type"] = valid_products["title"].apply(get_product_type)
valid_products["is_excluded"] = valid_products["title"].apply(is_excluded)

relevant_products = valid_products[
    valid_products["product_type"].notna() &
    (~valid_products["is_excluded"])
].copy()

relevant_products = relevant_products.drop(columns=["is_excluded"])

print("\nRelevant products by type:")
print(relevant_products["product_type"].value_counts())


Relevant products by type:
product_type
headphone    3792
laptop       2878
tablet       1395
keyboard     1047
mouse         844
phone         182
Name: count, dtype: int64


In [7]:
# balance sampling per category

products_per_type = 150  # adjust value count per type
core_products = (
    relevant_products
    .groupby("product_type", group_keys=False)[relevant_products.columns]
    .apply(lambda group: group.sample(n=min(len(group), products_per_type), random_state=42))
    .reset_index(drop=True)
)

print("\nCore products selected:", len(core_products))
print(core_products["product_type"].value_counts())

core_products["category_list"] = core_products["categories"].apply(ast.literal_eval)
core_products["main_category"] = core_products["category_list"].apply(
    lambda x: x[0][-1] if x and x[0] else None
)




Core products selected: 900
product_type
headphone    150
keyboard     150
laptop       150
mouse        150
phone        150
tablet       150
Name: count, dtype: int64


In [8]:
# realed products 
available_asins = set(relevant_products["asin"])

def extract_related_asins(value):
    if pd.isna(value):
        return []
    try:
        related_data = ast.literal_eval(value)
        if not isinstance(related_data, dict):
            return []
        result = []
        for asin_list in related_data.values():
            if isinstance(asin_list, list):
                result.extend(asin_list)
        return result
    except (ValueError, SyntaxError):
        return []

related_asins = set()
for value in core_products["related"]:
    for asin in extract_related_asins(value):
        if asin in available_asins:
            related_asins.add(asin)

related_asins = list(related_asins)[:100]

related_products = relevant_products[relevant_products["asin"].isin(related_asins)].copy()
related_products["category_list"] = related_products["categories"].apply(ast.literal_eval)
related_products["main_category"] = related_products["category_list"].apply(
    lambda x: x[0][-1] if x and x[0] else None
)

print("\nRelated products added:", len(related_products))



Related products added: 100


In [9]:
#combine

final_df = pd.concat([core_products, related_products], ignore_index=True)
final_df = final_df.drop_duplicates(subset=["asin"]).reset_index(drop=True)

print("\nFinal dataset shape:", final_df.shape)
print("Duplicate ASINs:", final_df["asin"].duplicated().sum())
print("\nFinal product_type distribution:")
print(final_df["product_type"].value_counts(dropna=False))


Final dataset shape: (995, 12)
Duplicate ASINs: 0

Final product_type distribution:
product_type
headphone    181
laptop       166
tablet       166
mouse        166
keyboard     165
phone        151
Name: count, dtype: int64


In [10]:
#Final columns — product_type now included, unlike the original
# dataset. Also decode any lingering HTML entities in text fields
# (&amp;, &quot;, &reg;) — previously flagged, unfixed issue,
# closed here at the source.

import html

final_df["title"] = final_df["title"].apply(lambda t: html.unescape(str(t)))
final_df["description"] = final_df["description"].apply(lambda t: html.unescape(str(t)))

final_df = final_df[[
    "asin", "title", "description", "categories", "price",
    "related", "imUrl", "brand", "product_type",
]].rename(columns={"imUrl": "image_url"})

print("\nFinal columns:", final_df.columns.tolist())
print("Missing values:\n", final_df.isnull().sum())


Final columns: ['asin', 'title', 'description', 'categories', 'price', 'related', 'image_url', 'brand', 'product_type']
Missing values:
 asin             0
title            0
description      0
categories       0
price            0
related         69
image_url        0
brand            0
product_type     0
dtype: int64


In [16]:
# Save the final recommendation dataset.
# The original raw dataset remains untouched.

output_path = "../processed/products.csv"

final_df.to_csv(
    output_path,
    index=False
)

print("Final dataset saved successfully.")
print("Location:", output_path)

Final dataset saved successfully.
Location: ../processed/products.csv
